# Adapter-Generator

Aufbau dieses Notebooks:

0. Importieren verwendeter Bibliotheken und Festlegen der Experiment-Parameter
1. Laden und splitten des verwendeten Datensatzes
2. Laden des Backbone-Modells
3. Initialisieren und Trainieren des Adapters 
4. Evaluation des trainierten Adapters auf manuell geprüften Test-Split und speichern des Adapters und der Testergebnisse
5. Falls ein ungefilterter, synthetischer Datensatz verwendet wurde: Datensatz filtern (Teil der Datensatz-Generierung)

---
## Schritt 0: Importieren verwendeter Bibliotheken und Festlegen der Experiment-Parameter

In [ ]:
import torch
import numpy as np
import random
import os
import re
import json

from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.models import Router, Normalize

from datasets import load_dataset, concatenate_datasets
import adapters
from adapters import AdapterConfig

from transformers import set_seed
from huggingface_hub import login

**Login to Hugging Face**

Used for pushing model to the Hugging Face Hub and downloading gated models or datasets

In [ ]:
login(token='eigenen Token einfügen', add_to_git_credential=True)

## Test Parameter

Zu Beginn werden alle Testlauf-spezifischen Parameter gesetzt.
Somit muss nur die folgende Code-Zelle für die Erstellung des gewünschten Adapters angepasst werden.

In [ ]:
# Hugging Face model ID
# Es wurden die Modell-Familie intfloat/multilingual-e5 und castorini/mdpr-tied-pft-msmarco verwendet
MODEL_ID = "intfloat/multilingual-e5-large"

# Festlegen welcher Trainings- & Validierungs-Datensatz für das Training des neuen Adapters genutzt werden soll:
# 1 = manuell erstellter Trainings- & Validierungs-Datensatz
# 2 = ungefilterter synthetischer Trainings- & Validierungs-Datensatz
# 3 = gefilterter synthetischer Trainings- & Validierungs-Datensatz
TRAIN_DATA_CASE = 2

# Festlegen ob Adapter auch für den Dokument-Encoder genutzt werden soll (siamesischer Bi-Encoder)
# oder ob der Adapter ausschließlich für den Query-Encoder genutzt werden soll (asymmetrischer Bi-Encoder)
USE_ASYMMETRIC_ENCODER = False

# Festlegen des Reduktions-Faktors
# Im Rahmen der Experimente wurde stets der Reduktions-Faktor 16 genutzt
REDUCTION_FACTOR = 16

# Definieren des Seeds. Genutzt wurden folgender Seed: 314
# Seed basieren auf den ersten 3 Stellen von π
SEED = 314

# Menge der Metrikwerte, die nach erfolgter Modell-Evaluation ausgegeben werden sollen
# Auswahl beschränkt auf die hierbei von SentenceTransformer berechneten Metrik-Scores und entspricht nicht der Metrik-Auswahl aus der Thesis
METRICS = [
    "map@100",
    "recall@1","recall@3","recall@5","recall@10"
]

Auf basis der angegebenen User-Parameter werden im folgenden Code-Block die zu verwendenden Pfade gesetzt

In [ ]:
def safe_name(s: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '__', s.strip('/'))

backbone_name = safe_name(MODEL_ID)

match TRAIN_DATA_CASE:
    case 1:
        target_location_name = "manual"
    case 2:
        target_location_name = "unfiltered_synthetic"
    case 3:
        target_location_name = "filtered_synthetic"

# Verzeichniss in dem der trainierte Adapeter abgespeichert werden soll
out_adapter_location = "adapters/" + target_location_name

# Verzeichnisse in denen die Adapter-Evaluierungs Ergebnisse abgespeichert werden (aggregiert & predictions)
out_results_aggregated = "results/aggregated/" + target_location_name
out_results_predictions = "results/predictions/" + target_location_name

# Verzeichniss des zu verwendenden Trainings- und Evaluierungsdatensatzes.
# Da der gefilterte Datensatz abhängig von dem Modell ist, wird in diesem Fall der Pfad individuell angepasst
if TRAIN_DATA_CASE == 3:
    if USE_ASYMMETRIC_ENCODER == True:
        dataset_location = "datasets/GerManualDPR/" + target_location_name + "_asymmetric/" + backbone_name
    else:
        dataset_location = "datasets/GerManualDPR/" + target_location_name + "/" + backbone_name
else:
    dataset_location = "datasets/GerManualDPR/" + target_location_name

# Erstellen die Verzeichnisse falls nicht vorhanden
os.makedirs(out_adapter_location, exist_ok=True)
os.makedirs(out_results_aggregated, exist_ok=True)
os.makedirs(out_results_predictions, exist_ok=True)
os.makedirs(dataset_location, exist_ok=True)

## Reproduzierbarkeit sicherstellen

Auf Basis des nun bestimmten SEED-Parameters sollen alle im Folgenden ausgeführten Operationen, derren Durchführung und Ergebniss von zufällig erzeugten Werten abhängen reproduzierbar gemacht werden. Hierbei ist zu beachten, dass das Experiment für die Arbeit auf MPS (Apple) durchgeführt wurde und nicht auf CUDA (Nvidia).

In [ ]:
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Keine CUDA-spezifischen Seed-Einstellungen, da keine NVIDIA-Grafigkarte im Rahmen der Arbeit genutzt wurde.

---
## Schritt 1: Trainingsdatensatz splitten & vorbereiten

In diesem Schritt wird der zu verwendende Trainingsdatensatz zufällig in einen Train- und Validate-Split im 90/10 Verhältnis aufgeteilt.
Hierbei wird auch die Reihenfolge der Einträge verändert. Wird für das Training der manuell erstellte Datensatz verwendet, werden stattdessen die abgelegten Splits verwendet.

Für das Training über den [SentenceTransformerTrainer](https://sbert.net/docs/package_reference/sentence_transformer/trainer.html) bzw. die Evaluierung über den [InformationRetrievalEvaluator](https://sbert.net/docs/package_reference/sentence_transformer/evaluation.html#sentence_transformers.evaluation.InformationRetrievalEvaluator) müssen die Datensätze passend vorbereittet werden. Hierbei werden drei Schlüssel-Datenstrukturen verlangt:

- Ein corpus dictionary welches IDs zu den Chunks mapped und keine Duplikate enthält (`{chunk_id: text_chunk}`)
- Ein queries dictionary welches IDs zu den Fragen mapped (`{query_id: question_text}`)
- Ein relevant_docs_test dictionary welches spezifiziert, welche corpus Chunks für die jeweilige Frage passenend sind (`{query_id: [relevant_chunk_id]}`)

Hinweis: Für das Relevanz-Mapping wird die chunk_id als connecting key zwischen der Frage und dem relevanten Chunk genutzt.

In [ ]:
# Laden aller Datensatz-Splits aus den entsprechenden JSON-Datein
# Diese enthalten Paare aus Fragen (anchors) und dem dazugehörigen Text-Chunks (positives)
if TRAIN_DATA_CASE == 1:
    train_dataset = load_dataset("json", data_files=f'{dataset_location}/train_dataset.json', split="train")
    validate_dataset = load_dataset("json", data_files=f'{dataset_location}/validate_dataset.json', split="train")
else:
    full_train = load_dataset("json", data_files=f"{dataset_location}/train_dataset.json", split="train")
    split = full_train.train_test_split(test_size=0.1, seed=SEED, shuffle=True)
    train_dataset = split["train"]
    validate_dataset = split["test"]   # 10% als Validation
print ("Dataset loaded from directory: " + dataset_location)

# Für die Evaluierung wird stets der geprüfte GerManualDPR-Test-Split genutzt
test_dataset = load_dataset("json", data_files="datasets/GerManualDPR/manual/test_dataset.json", split="train")

# Lade des gesamten Coprus aus der entsprechenden JSON-Datei
corpus_dataset = load_dataset("json", data_files="datasets/GerManualDPR/full_corpus.json", split="train")

# Konvertieren vom datasets in das dictionary Format. Dieses wird vom InformationRetrievalTrainer & -Evaluator benötigt.
# corpus: Mapped die chunk_id zu den Chunks (Passagen).
# Format: {chunk_id: text_chunk}
corpus = dict(
    zip(corpus_dataset["chunk_id"], corpus_dataset["positive"])
)

# test_queries: Mapped die query IDs zu den jeweiligen Fragen
# Format: {query_id: question_text}
test_queries = dict(
    zip(test_dataset["id"], test_dataset["anchor"])
)
validate_queries = dict(
    zip(validate_dataset["id"], validate_dataset["anchor"])
)

# Erstellung eines Mappings zwischen den Fragen und dem dazugehörigen Chunk für die Validierung und den Test
# relevant_docs: Query-ID -> relevante Chunk-ID
relevant_docs_test = {qid: [chk] for qid, chk in zip(test_dataset["id"], test_dataset["chunk_id"])}
relevant_docs_validate = {qid: [chk] for qid, chk in zip(validate_dataset["id"], validate_dataset["chunk_id"])}

Für das Training und die Evaluierung werden InformationRetrievalEvaluator-Instanzen verwendet. Im Rahmen des Experiments wird der manuell erstellte Test-Split ausschließlich für die Evaluierung verwendet. Für das Training wird daher eine separate InformationRetrievalEvaluator-Instanz basierend auf dem Validate-Split erstellt.

Diese enthalten alle Chunks des Datensatzes (corpus), die Test- bzw. Validate-Fragen (queries) und die Verknüpfung zwischen den Fragen und dem relevanten Chunk.

Hierbei wird bereits die zu verwendende Ähnlichkeits-Funktion angegeben: Kosinusähnlichkeit.

In [ ]:
# Initialisierung des Evaluators
evaluator = InformationRetrievalEvaluator(
    queries=test_queries,
    corpus=corpus,
    relevant_docs=relevant_docs_test,
    name="myEvaluator",
    score_functions={"cosine": cos_sim},
    write_predictions=True,               # <- schreibt Top-k/Predictions als JSONL
)

# Initialisierung des Validierers
validator = InformationRetrievalEvaluator(
    queries=validate_queries,
    corpus=corpus,
    relevant_docs=relevant_docs_validate,
    name="myValidator",
    score_functions={"cosine": cos_sim},
)

---
## Schritt 2: Backbone-Modell laden

In diesem Schritt wird das zu beginn definierte Backbone-Modell geladen. 

Auf Basis der definierten Parameter des aktuellen Experimentdurchlaufs wird der verwendete Bi-Encoder nur aus einer Modell-Instanz oder mittels eines [Routers](https://sbert.net/docs/package_reference/sentence_transformer/models.html#sentence_transformers.models.Router) aus zwei separaten Modell-Instanzen aufgebaut. Die Nutzung des Routers ermöglicht eine asymmetrische Bi-Encoder-Architektur, indem nur eine der Modell-Instanzen mittels eines Adapters angepasst werden kann.

In [ ]:
# Laden des Backbone-Modells über SentenceTransformer
st_model = SentenceTransformer(
    MODEL_ID, 
    device="cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
)

# Falls nur der Query-Encoder mittels Adapter angepasst wird, wird mittels eines Routers eine asymmetrische
# Modell-Instanz erzeugt und die Parameter des Doc-Encoders gefreezed.
if USE_ASYMMETRIC_ENCODER == True:
    st_model_doc = SentenceTransformer(
        MODEL_ID, 
        device="cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
    )

    # Router bauen
    router = Router.for_query_document(
        query_modules=list(st_model.children()),
        document_modules=list(st_model_doc.children()),
    )

    st_model_asymmetric = SentenceTransformer(modules=[router, Normalize()])

    # Freezen aller Parameter des Doc-Encoders
    for param in st_model_doc.parameters():
        param.requires_grad = False

    st_model_doc_total_params = sum(p.numel() for p in st_model_doc.parameters())
    st_model_doc_frozen_params = sum(p.numel() for p in st_model_doc.parameters() if p.requires_grad == False)

    print(f"Total Parameters of doc-Encoder:   {st_model_doc_total_params}")
    print(f"Frozen Parameters of doc-Encoder:  {st_model_doc_frozen_params}")
    print(f"Frozen share:  {st_model_doc_frozen_params / st_model_doc_total_params:.2%} \n")

print (f"In your case following device is used: {st_model.device}")

Falls noch keine Baseline-Ergebniss zu dem Backbone-Modell ohne Adapter auf den manuell erstellten GerManualDPR-Test-Split abliegen, wird im folgenden Code-Block zudem eine Evaluierung dieses Modells durchgeführt und die Ergebnisse abgespeichert.

In [ ]:
# Wenn noch keine Baseline-Ergebnisse für den aktuellen Seed existieren -> Baseline-Evaluation durchführen ---
results_outfile = os.path.join("results", "aggregated", f"baseline_{backbone_name}.json")
prediction_outpath = os.path.join("results", "predictions", "baseline", backbone_name)

# Erstellen des Out-Verzeichnisses falls nicht vorhanden
os.makedirs(prediction_outpath, exist_ok=True)

if os.path.exists(results_outfile):
    with open(results_outfile, "r", encoding="utf-8") as f:
        base_results = json.load(f)
    print("Geladene Ergebnisse aus:", results_outfile)
else:
    print("Evaluierung wird durchgeführt!")
    base_results = evaluator(st_model, output_path=prediction_outpath)

# Name des Evaluators und Embedding-Dimension
eval_name = getattr(evaluator, "name", "full_dim")
try:
    emb_dim = st_model.get_sentence_embedding_dimension()
except Exception:
    emb_dim = None

print("\nBase Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} Base Model")
print("-" * 85)

for metric in METRICS:
    key = f"{eval_name}_cosine_{metric}"   # z.B. "full_dim_cosine_ndcg@10"
    val = base_results.get(key, float("nan"))
    metric_name = f"=={metric}==" if metric == "map@100" else metric
    print(f"{metric_name:15} {val:12.4f}")

print("-" * 85)
print(f'The encoder uses {emb_dim} dimensions')

# Ergebnisse lokal abspeichern
with open(results_outfile, "w", encoding="utf-8") as f:
    json.dump(base_results, f, indent=2, ensure_ascii=False)
print("✓ results saved:", results_outfile)

---
## Schritt 3 Adapter initialisieren & trainieren

Zunächst wird ein Pfeiffer-Adapter initialisiert und in das Modell eingesetzt. Hierfür wird [`adapters`](https://docs.adapterhub.ml/quickstart.html#introduction) verwendet.

In [ ]:
# Fügt Adapter in ST-Modell ein
hf_model = st_model.transformers_model
adapters.init(hf_model)

name = f"pfeiffer_r{REDUCTION_FACTOR}"
cfg = AdapterConfig.load("pfeiffer", reduction_factor=REDUCTION_FACTOR)
hf_model.add_adapter(name, cfg)
target_name = f"pfeiffer_r{REDUCTION_FACTOR}"
hf_model.train_adapter(target_name)           # freeze backbone, train only this adapter
hf_model.set_active_adapters(target_name)     # activate exactly this adapter

print("Active adapter:", hf_model.active_adapters)
print(sum(p.requires_grad for p in hf_model.parameters()), "trainable tensores")

Für das Training wird Sentence Transformers [`SentenceTransformerTrainer`](https://sbert.net/docs/sentence_transformer/training_overview.html) verwendet.
Hierfür werden zunächst die Loss-Funktion und die Trainingsparameter definiert.

Als Loss-Funktion wird MultipleNegativesRankingLoss genutzt.
Diese Loss-Funktion entspricht – laut [Dokumentation](https://sbert.net/docs/package_reference/sentence_transformer/losses.html?#multiplenegativesrankingloss) – dem InfoNCE/SimCSE-Ansatz und ist somit Teil der NLL-Loss-Familie

In [ ]:
# Loss-Function
train_loss = MultipleNegativesRankingLoss(st_model)

# Trainingparameter
if USE_ASYMMETRIC_ENCODER == True:
    args = SentenceTransformerTrainingArguments(
        output_dir=f'training_output/{target_name}_{backbone_name}',
        num_train_epochs=20,
        per_device_train_batch_size=5, 
        gradient_accumulation_steps=6,
        per_device_eval_batch_size=5,
        learning_rate=1e-4,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        eval_strategy="steps",
        eval_steps=100,
        logging_steps=100,
        save_total_limit=3,
        seed=SEED,
        router_mapping={
            "anchor": "query",
            "positive": "document",
            "negative": "document",
        },
    )
else:
    args = SentenceTransformerTrainingArguments(
        output_dir=f'training_output/{target_name}_{backbone_name}',
        num_train_epochs=20, # Für die Epochenanalyse der Backbone-Modelle wurde dieser Wert angepasst
        per_device_train_batch_size=5,
        gradient_accumulation_steps=6,
        per_device_eval_batch_size=5,
        learning_rate=1e-4,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        eval_strategy="steps",
        eval_steps=100,
        logging_steps=100,
        save_total_limit=3,
        seed=SEED,
    )

Damit sind nun alle benötigten Eingabeparameter für den `SentenceTransformerTrainer` vorbereitet und können übergeben werden.
Daraufhin wird das Training des Adapters iniziiert.

In [ ]:
trainer = SentenceTransformerTrainer(
    model=st_model,
    args=args,
    train_dataset=train_dataset.select_columns(["anchor", "positive"]),
    eval_dataset =validate_dataset.select_columns(["anchor", "positive"]),
    loss=train_loss,
    evaluator=validator,
)

# Trainig starten
trainer.train()

---
## Schritt 4: Adapter testen und speichern

Zunächst wird der soeben trainierte Adapter lokal abgespeichert.

In [ ]:
# Speichern das Adapters
if USE_ASYMMETRIC_ENCODER == True:
    save_dir = f'{out_adapter_location}/{target_name}_{backbone_name}__S{SEED}_asymmetric'
else:
    save_dir = f'{out_adapter_location}/{target_name}_{backbone_name}__S{SEED}'

hf_model.save_adapter(save_dir, target_name)
print("✓ adapter saved::", save_dir)

Abschließend wird der Adapter auf dem manuell erstellten Test-Split evaluiert.

Für die Evaluierung wird eine neue SentenceTransformer-Instanz erstellt und der abgespeicherte Adapter neu geladen. Damit wird sichergestellt, dass die Evaluation auf dem tatsächlich persistierten Modellzustand basiert und nicht auf einem noch im Arbeitsspeicher befindlichen Trainingszustand.

Hierbei werden die predictions lokal abgespeichert, um basierend darauf die Metriken und derren Signifikanz berechnen zu können. Die von Sentence-Transformers berechneten Metrik-Werte werden separat abgespeichert, um damit die selbst berechneten Metrik-Werten überprüfen zu können.

Die von Sentence-Transformers berechneten Metrik-Werte werden in der Ausgabe jenen des Backbone-Modells gegenübergestellt.

In [ ]:
backbone_with_adapter = SentenceTransformer(
    MODEL_ID, device="cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)

ft_hf = backbone_with_adapter.transformers_model
adapters.init(ft_hf)
ft_hf.load_adapter(save_dir, load_as="pfeiffer_adapter")  # Adapter aus dem soeben gespeicherten Ordner
ft_hf.set_active_adapters("pfeiffer_adapter")

print("Loaded adapter from:", save_dir)
print("Activated adapter:", ft_hf.active_adapters)

# Falls Adapter nur auf Query-Encoder trainiert wurde (asymmetrisch) muss für die Evaluation mittels
# Router ein asymmetrisches Modell mit Baseline als Doc-Encoder geladen werden
if USE_ASYMMETRIC_ENCODER == True:
    baseline_model = SentenceTransformer(
        MODEL_ID, 
        device="cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
    )

    # Router bauen
    router = Router.for_query_document(
        query_modules=list(backbone_with_adapter.children()),
        document_modules=list(baseline_model.children()),
    )

    asymmetric_model = SentenceTransformer(modules=[router, Normalize()])

# Pfad für die Ablage der Prediction anpassen
if USE_ASYMMETRIC_ENCODER == True:
    prediction_outpath = os.path.join(out_results_predictions, f"{target_name}_{backbone_name}__S{SEED}_asymmetric")
else:
    prediction_outpath = os.path.join(out_results_predictions, f"{target_name}_{backbone_name}__S{SEED}")
    
os.makedirs(prediction_outpath, exist_ok=True)

# Evaluiere das optimierte Modell
if USE_ASYMMETRIC_ENCODER == True:
    ft_results = evaluator(asymmetric_model, output_path=prediction_outpath)
else:
    ft_results = evaluator(backbone_with_adapter, output_path=prediction_outpath)

# Vergleich: Adapter (ft_results) vs. Base (base_results)
eval_name = getattr(evaluator, "name", "full_dim")

def get_val(res, m):
    return res.get(f"{eval_name}_cosine_{m}", float("nan"))

print("\nBase Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'Adapter':>10} {'Base-Model':>12} {'Abs. Improvement':>16} {'% Improvement':>14}")
print("-" * 85)

for metric in METRICS:
    a = get_val(ft_results, metric)
    b = get_val(base_results, metric)
    abs_impr = a - b
    pct_impr = (abs_impr / b * 100.0) if (b not in (0.0, float('nan'))) else float('nan')

    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15} {a:10.4f} {b:12.4f} {abs_impr:16.4f} {pct_impr:13.2f}%")

print("-" * 85)

if USE_ASYMMETRIC_ENCODER == True:
    results_outfile = os.path.join(out_results_aggregated, f"{target_name}_{backbone_name}__S{SEED}_asymmetric.json")
else:
    results_outfile = os.path.join(out_results_aggregated, f"{target_name}_{backbone_name}__S{SEED}.json")
    
with open(results_outfile, "w", encoding="utf-8") as f:
    json.dump(ft_results, f, indent=2, ensure_ascii=False)
    
print("✓ aggregated results saved:", results_outfile)
print("✓ prediction results saved at:", prediction_outpath)

---
## Schritt 5: Filtern des synthetischen Datensatzes

Falls ein ungefilterter synthetischer Datensatz genutzt wurde wird nun in einem letzten Schritt mittels des soeben trainierten Adapters eine Filterung des Datensatzes durchgeführt. Dieser Schritt ist Teil der Datensatzgenerierung.

In [ ]:
from functions.promptagator_filter import filter_ai_dataset_topk

# Filtern nur ausführen, wenn aktueller Adapter auf den ungefilterten, synthetischen Datensatz trainiert wurde
if TRAIN_DATA_CASE == 2:
    st_model = asymmetric_model if USE_ASYMMETRIC_ENCODER else backbone_with_adapter
    
    filter_ai_dataset_topk(
        st_model=st_model,
        corpus=corpus,
        dataset_location=dataset_location,
        max_required_rank=10,
        device=backbone_with_adapter.device,
        asymmetric_encoder=USE_ASYMMETRIC_ENCODER,
        backbone_name=backbone_name,
        doc_batchsize=128,
        q_batchsize=64,
    )